# Working Memory of Multi-Object Scenes in Primate Frontal Cortex (Watters, Gabel, Tenenbaum & Jazayeri 2026): reproduction on DANDI 000620

**Paper.** N. Watters, J. Gabel, J. Tenenbaum, M. Jazayeri, "Working Memory of Multi-Object Scenes in Primate Frontal Cortex", bioRxiv preprint version 1, doi:10.64898/2026.01.27.702062. The claims file (`claims/000620/10.64898-2026.01.27.702062.yaml`) was written from the bioRxiv full text (main text, Methods, legends of Extended Data Figures 1 to 11) and the paper's code, https://github.com/jazlab/multi_object_memory_2025 at commit 6e4b333b78cd4461e6373a42378535e8fd392cdb, which this attempt also reads for method details (cited by file in `wm.py` and below).

**Dataset.** DANDI 000620, "Multi-object working memory in macaque frontal cortex", version **0.261019.2000** (doi:10.48324/dandi.000620/0.261019.2000), streamed with remfile and h5py. Why this version: the paper cites the dandiset without a version, so the most recent published version is pinned. The four published versions (0.260127.2208, 0.261014.1424, 0.261019.2000 and the draft) have the same 142 assets and the same total size (5,490,107,572,879 bytes), and the release notes of 0.261019.2000 say "No changes to data", so no earlier version matches the claims file's description better. Each session has three files: raw ecephys (5.5 TB in all, not used), `spikesorting` (units with spike times, quality, probe, stable intervals and per-trial stability) and `behavior+task` (the trials table, eye position and reward line). This notebook uses the trials tables and the units tables.

**What is tested where.** This notebook tests the behaviour, unit-count, selectivity and correlation claims. The decoding claims (ext3ab-decoding, ext3c-decoder-orthogonality) are in `decoding_watters_2026.ipynb`, and variants of its decoders that are not the code's in `decoding_variants_watters_2026.ipynb`. The claims that need the paper's encoding models fitted (13 claims) are blocked as `too_large`; the last section times the computation.

| claim | where | status |
|---|---|---|
| fig1b-performance-load | this notebook | tested |
| fig1c-reaction-times | this notebook | tested |
| ext1-misbinding-errors | this notebook | tested |
| triangle-unit-counts | this notebook | tested |
| ring-unit-counts | this notebook | tested |
| ext2-stimulus-delay-activity | this notebook | tested |
| ext3ab-decoding | decoding notebook | tested |
| ext3c-decoder-orthogonality | decoding notebook | tested |
| fig2b-selectivity-range | this notebook | tested |
| fig2d-signal-noise-triangle | this notebook | tested |
| fig5ab-ring-behavior | this notebook | tested |
| fig5cd-ring-tuning | this notebook | tested |
| fig5e-signal-noise-ring | this notebook | tested |
| fig4a-gain-best-triangle | this notebook, timing only | blocked, too_large |
| ext6-gain-best-areas | timing only | blocked, too_large |
| ext5-general-slot | timing only | blocked, too_large |
| fig4b-identifiability | timing only | blocked, too_large |
| nslots-two-best | timing only | blocked, too_large |
| ext11-kappa-tuning | timing only | blocked, too_large |
| ext7-gains-distributed | timing only | blocked, too_large |
| fig4c-gain-performance-triangle | timing only | blocked, too_large |
| fig4d-gain-rt-triangle | timing only | blocked, too_large |
| ext8-slot-switching-behavior | timing only | blocked, too_large |
| fig5f-gain-best-ring | timing only | blocked, too_large |
| fig5g-gain-performance-ring | timing only | blocked, too_large |
| fig5h-gain-rt-ring | timing only | blocked, too_large |

Shared definitions T1 to T8 are those of the claims file's opening comment; `wm.py` says where each parameter comes from.


In [ ]:
import warnings
warnings.filterwarnings("ignore")
import time
import numpy as np, pandas as pd
import wm
t_start = time.time()
assets = wm.list_assets()
print(f"Dandiset {wm.DANDISET} version {wm.VERSION}: {len(assets)} assets,", assets.kind.value_counts().to_dict())
print("Raw ecephys files (not used):", f"{assets[assets.kind == 'ecephys']['size'].sum() / 1e12:.2f} TB")
keys = sorted(set(zip(assets.subject, assets.session)))
S = [wm.load_session(assets, subj, sess) for subj, sess in keys]
print(f"Loaded {len(S)} sessions in {time.time() - t_start:.0f} s")


In [ ]:
import numpy as np, pandas as pd, scipy.stats as st
import matplotlib
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
MONKEY = {"Perle": "P", "Elgar": "E"}
for x in S:
    x["trials"]["subject"] = x["subject"]; x["trials"]["session"] = x["session"]
    x["trials"]["task"] = x["task"]; x["trials"]["trial"] = np.arange(len(x["trials"]))
# Physiology sessions of each dataset: sessions with spike sorting. Ring sessions are those
# with 1- and 2-object trials (Methods, Behavioral Task); Perle 2022-06-06 has 2-object
# trials only and is kept apart (see the data check).
tri = [x for x in S if x["task"] == "triangle" and x["units"] is not None]
ring = [x for x in S if x["task"] == "ring" and x["units"] is not None
        and (x["trials"].n_obj == 1).any()]
other = [x for x in S if x not in tri and x not in ring]
T = pd.concat([x["trials"] for x in S], ignore_index=True)
T["physiology"] = T.session.isin([x["session"] for x in tri + ring])
print("Triangle sessions:", {m: sum(x["subject"] == m for x in tri) for m in MONKEY})
print("Ring sessions:", {m: sum(x["subject"] == m for x in ring) for m in MONKEY})
print("Other sessions:", [(x["subject"], x["session"], x["task"], x["units"] is not None,
                           sorted(x["trials"].n_obj.unique())) for x in other])


## Data check

Printed below: sessions, trials and units by subject, session and condition; ranges and units of every variable used; the reward against the response error (which correctness radius the task used); and the clock check (location information appears after stimulus onset and in the delay, and not before onset or with shifted labels).

In [ ]:
rows = []
for x in S:
    tr, u = x["trials"], x["units"]
    c = tr[tr.completed]
    rows.append(dict(
        monkey=MONKEY[x["subject"]], session=x["session"], task=x["task"],
        trials=len(tr), completed=len(c),
        n1=(tr.n_obj == 1).sum(), n2=(tr.n_obj == 2).sum(), n3=(tr.n_obj == 3).sum(),
        on_tri=round(tr.on_triangle.mean(), 3),
        units=np.nan if u is None else len(u),
        good=np.nan if u is None else (u.quality == "good").sum(),
        DMFC=np.nan if u is None else (u.area == "DMFC").sum(),
        FEF=np.nan if u is None else (u.area == "FEF").sum()))
inv = pd.DataFrame(rows)
print(inv.to_string(index=False))
print("\nQuality labels:", sorted(set(np.concatenate([x["units"].quality.unique() for x in S if x["units"] is not None]))))
print("Probes:", sorted(set(np.concatenate([x["units"].probe.unique() for x in S if x["units"] is not None]))))
c = T[T.completed]
def rng_(v): return f"min {np.nanmin(v):.4f}, median {np.nanmedian(v):.4f}, max {np.nanmax(v):.4f}"
print("\nPhase durations of completed trials (s):")
print("  stimulus (stimulus to delay onset):", rng_(c.phase_delay_time - c.phase_stimulus_time))
print("  delay (delay to cue onset):        ", rng_(c.phase_cue_time - c.phase_delay_time))
print("  cue (cue to response-phase onset): ", rng_(c.phase_response_time - c.phase_cue_time))
print("  reaction time (response_time - cue onset):", rng_(c.reaction_time))
print("  share of completed trials with reaction time < 0.18 s:", round(float((c.reaction_time < 0.18).mean()), 4))
print("  share with reaction time < cue duration (0.25 s):", round(float((c.reaction_time < 0.25).mean()), 4))
r_obj = np.hypot(T[["x0", "x1", "x2"]].values, T[["y0", "y1", "y2"]].values)
print("\nObject eccentricity (arena widths):", rng_(r_obj), "; expected 0.35 = 8.4 deg / 24 deg")
print("Response position x, y (arena widths from centre):", rng_(c.resp_x), "|", rng_(c.resp_y))
print("Identities:", sorted(set(T.id0.dropna()) | set(T.id1.dropna()) | set(T.id2.dropna())))
print("delay_object_blanks true:", int(T.delay_object_blanks.sum()), "; nonzero object velocities:", int((T.max_speed > 0).sum()),
      "; trials without exactly one target:", int((T.n_target_flags != 1).sum()))
tt = T[(T.task == "triangle") & T.on_triangle]
print("Distinct triangle object locations on on-triangle trials:", sorted(set(tt[["loc0","loc1","loc2"]].values.ravel()) - {-9}))
print("Units: spikes unsorted in", sum(x["units"].attrs["n_unsorted_units"] for x in S if x["units"] is not None),
      "units; spikes outside the stable intervals:", sum(x["units"].attrs["n_spikes_outside_intervals"] for x in S if x["units"] is not None))
allrates = np.concatenate([np.nanmean(x["rates"]["delay"], 1) for x in tri + ring])
print("Unit mean delay-phase rate (Hz):", rng_(allrates), "; units with no completed stable trial:", int(np.isnan(allrates).sum()))
print("closed-loop minus post-hoc response time (s):", rng_(c.closed_loop_response_time - c.response_time))
# Clock check: for each physiology session, the share of units whose rate differs across the
# three object locations on 1-object trials (Kruskal-Wallis p < 0.01), before stimulus onset
# (-300 to 0 ms), early in the stimulus (50 to 350 ms) and in the delay, and the last two with
# the labels rolled by one trial (a control that keeps the label counts).
clk = []
for x in tri + ring:
    tr = x["trials"]
    m = (tr.n_obj == 1).values & tr.completed.values
    if x["task"] == "triangle":
        m &= tr.on_triangle.values
        lab = tr.loc0.values
    else:
        lab = wm.ring_sector(tr.theta0.values)
    res = {}
    for w, shift in [("pre", 0), ("early", 0), ("early", 1), ("delay", 0), ("delay", 1)]:
        R = x["rates"][w][:, m]
        L = np.roll(lab[m], shift)
        sig = []
        for r in R:
            ok = np.isfinite(r)
            g = [r[ok & (L == k)] for k in range(3)]
            if min(len(v) for v in g) < 5 or np.ptp(r[ok]) == 0:
                continue
            sig.append(st.kruskal(*g).pvalue < 0.01)
        res[f"{w}{'_shifted' if shift else ''}"] = np.mean(sig) if sig else np.nan
    clk.append(dict(monkey=MONKEY[x["subject"]], session=x["session"], task=x["task"], **res))
clk = pd.DataFrame(clk)
print("\nClock check, share of units location-modulated (Kruskal-Wallis p < 0.01):")
print(clk.round(3).to_string(index=False))
print("Sessions where early modulation is not above both controls:",
      clk[(clk.early <= clk.pre) | (clk.early <= clk.early_shifted)].session.tolist())
print("Sessions where delay modulation is not above the shifted control:",
      clk[clk.delay <= clk.delay_shifted].session.tolist())
# Correctness radius (T2): which radius did the task reward? Reward against the closed-loop
# response error (the position the task used for reward), completed trials of all sessions.
c = T[T.completed]
bins = [0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 10]
tab = c.groupby(pd.cut(c.target_err_closed_loop, bins, right=False), observed=True).rewarded.agg(["size", "mean"])
tab.columns = ["completed trials", "share rewarded"]
print("\nShare of completed trials rewarded, by closed-loop distance to the target (arena widths; 0.1 = 2.4 deg, 0.2 = 4.8 deg):")
print(tab.round(4).to_string())
print("\nElectrode locations named in the spikesorting files:",
      sorted(set(l for x in S if x["units"] is not None for l in x["units"].attrs["electrode_locations"])),
      "; the area of a unit is taken from its probe (s0 = DMFC Neuropixels, vprobe0/1 = FEF V-probes; claims file, code unit_filters.py).")
# Recordings deposited twice: compare trial timing (relative to the first trial) and the
# sorted spike counts of the units between every pair of sessions.
import hashlib
fp = {}
for x in S:
    tr = x["trials"]
    k = ("trials", hashlib.md5(np.round(tr.phase_stimulus_time.values - tr.phase_stimulus_time.values[0], 4).tobytes()).hexdigest())
    fp.setdefault(k, []).append(x["session"])
    if x["units"] is not None:
        k = ("units", hashlib.md5(np.sort(x["units"].n_spikes.values).tobytes()).hexdigest())
        fp.setdefault(k, []).append(x["session"])
print("Sessions sharing trial timing or sorted spike counts:", [v for v in fp.values() if len(v) > 1])


Checks of the metadata against the data. Sessions: 24 Triangle physiology sessions (P 10, E 14) and 22 Ring sessions (P 10, E 12), as the Methods state; two more Perle sessions are in the dandiset, 2022-06-06 (spike sorting, 2-object trials only, so not a Ring session by the Methods' design) and 2022-06-08 (behaviour only); both are left out. No two sessions share their trial timing or their sorted spike counts. Objects lie at eccentricity 0.35 arena widths (8.4 degrees), with three identities and, on the triangle, three locations, as described. The stimulus and delay phases last 1.000 s (median), as described, but the cue phase (cue onset to response-phase onset) lasts 0.250 s, where the Methods say 180 ms; 47.9% of completed trials have a post-hoc response time earlier than 0.25 s after cue onset, 1.1% earlier than 0.18 s, and the earliest is 12.5 ms before cue onset, so `response_time` reads as saccade onset (the closed-loop time is 134 ms later at the median). Every completed trial with a closed-loop error under 0.2 arena widths (4.8 degrees) was rewarded and none at 0.2 or more, so the task scored correctness at the code's radius, not the Methods' 2.4 degrees. The electrodes tables name no location ("unknown"); areas come from the probe. Spike times are sorted and all lie inside the units' stable intervals. Clock: in every session the share of location-modulated units in the delay exceeds the share with labels shifted by one trial; early in the stimulus it exceeds both controls in every session except Elgar 2022-08-23 (0.019 against 0.023 before onset), whose delay-phase modulation is above control (0.051 against 0.023). The trial table and the spike times are consistent with a shared clock in every session; for Elgar 2022-08-23 this rests on the delay window only.

## Behaviour

### fig1b-performance-load

> In both monkeys accuracy falls as the number of objects grows, yet stays far above the chance of saccading to a random stimulus object for two- and three-object stimuli.

Source: Results, Task, Behavior, and Hypotheses, paragraph 1; Figure 1b. Paper: "Animals learned the task and exhibited variable reaction times, with error rates increasing as the number of objects grew ... However, performance remained far above chance for two- and three-object stimuli"

Criterion (claims file): Per monkey, the fraction correct (T2) of completed trials (T1) for 1, 2 and 3 objects; chance is 1/n, as the Fig. 1b legend defines it. Reproduced when, for both monkeys, the fraction falls from 1 to 2 to 3 objects and the 2- and 3-object fractions lie above 1/2 and 1/3. The paper gives no test; "far above" is not given a number, so the margin is reported and not judged. Which trials form the figure is open: (a) Triangle trials on the triangle only, as the paper's figures/figure_1/performance/performance.ipynb plots; (b) Triangle and Ring trials pooled, as the legend says (the Ring has no 3-object trials). Each reading is run under both correctness radii of T2. For reference, that notebook printed, for reading (a) with the code's radius, P 0.953, 0.93, 0.692 and E 0.928, 0.759, 0.561 for 1, 2, 3 objects; these are reported against, not used to judge.

In [ ]:
# fig1b-performance-load. Completed trials (T1) of physiology sessions; correctness under both
# radii (T2). Reading (a): Triangle trials on the triangle; (b): those pooled with all Ring trials.
Tp = T[T.completed & T.physiology]
sel_a = Tp[(Tp.task == "triangle") & Tp.on_triangle]
sel_b = pd.concat([sel_a, Tp[Tp.task == "ring"]])
perf = []
for reading, d in [("a", sel_a), ("b", sel_b)]:
    for rad in wm.RADII:
        g = d.groupby(["subject", "n_obj"])[f"correct_{rad}"].agg(["mean", "size"]).reset_index()
        for _, r in g.iterrows():
            perf.append(dict(reading=reading, radius=rad, monkey=MONKEY[r.subject], n_obj=r.n_obj,
                             frac_correct=r["mean"], chance=1 / r.n_obj, n_trials=r["size"]))
perf = pd.DataFrame(perf)
perf["margin"] = perf.frac_correct - perf.chance
print(perf.round(3).to_string(index=False))
ok = {}
for (reading, rad), g in perf.groupby(["reading", "radius"]):
    res = []
    for m, gm in g.groupby("monkey"):
        f = gm.set_index("n_obj").frac_correct
        res.append(bool(f[1] > f[2] > f[3] and f[2] > 0.5 and f[3] > 1 / 3))
    ok[(reading, rad)] = all(res)
print("\nFalls 1 > 2 > 3 and above chance at 2 and 3, both monkeys:", ok)
fig, axes = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
for ax, m in zip(axes, ["P", "E"]):
    for (reading, rad), g in perf[perf.monkey == m].groupby(["reading", "radius"]):
        ax.plot(g.n_obj, g.frac_correct, "o-", label=f"{reading}, {rad}")
    ax.plot([1, 2, 3], [1, 1/2, 1/3], "k--", label="chance")
    ax.set_title(f"Monkey {m}"); ax.set_xlabel("objects"); ax.set_ylim(0, 1)
axes[0].set_ylabel("fraction correct"); axes[1].legend(fontsize=7); plt.show()
print("All plotted values lie in [0, 1].")


**Result: reproduced.** In both monkeys and under every reading and radius the fraction correct falls from 1 to 2 to 3 objects and the 2- and 3-object fractions are above 1/2 and 1/3. Reading (a) with the code's radius gives P 0.953, 0.930, 0.692 and E 0.928, 0.759, 0.561, the values the paper's notebook printed. The smallest margin above chance is 0.222 (E, 3 objects, Methods radius).

### fig1c-reaction-times

> Reaction times, from cue onset to the response saccade, vary from trial to trial, with trials faster than 180 ms counted as broken fixation.

Source: Results, Task, Behavior, and Hypotheses, paragraph 1; Figure 1c legend. Paper: "Animals learned the task and exhibited variable reaction times"

Criterion (claims file): Per monkey, reaction times (T3) of completed trials, correct and incorrect apart, as in figures/figure_1/reaction_times/reaction_times.ipynb, which bins them from 100 to 600 ms and shows 150 to 500 ms. Trials with a saccade earlier than 180 ms after cue onset are left out as broken fixation (Fig. 1c legend). Which trials form the figure is open, as for fig1b-performance-load: (a) Triangle trials on the triangle, as the notebook plots; (b) Triangle and Ring trials pooled, as the legend says; both are run. The paper fixes no measure of variability. Reproduced when the distribution of each monkey is not degenerate (interquartile range above one 10 ms bin) and its median lies inside the plotted 150 to 500 ms range; these cut-offs are the agent's reading of "variable" and are recorded as such. The histograms, quartiles and the share of trials under 180 ms before the cut are reported.

In [ ]:
# fig1c-reaction-times. Reaction time (T3) of completed trials, correct (code radius) and
# incorrect apart; trials under 180 ms are left out (Fig. 1c legend).
rt_rows = []
fig, axes = plt.subplots(2, 2, figsize=(8, 5), sharex=True)
for i, (reading, d) in enumerate([("a", sel_a), ("b", sel_b)]):
    for j, m in enumerate(["Perle", "Elgar"]):
        dm = d[d.subject == m]
        under = float((dm.reaction_time < 0.18).mean())
        dk = dm[dm.reaction_time >= 0.18]
        for lab, dd in [("all", dk), ("correct", dk[dk["correct_code_4.8deg"]]),
                        ("incorrect", dk[~dk["correct_code_4.8deg"]])]:
            q = np.percentile(dd.reaction_time, [25, 50, 75])
            rt_rows.append(dict(reading=reading, monkey=MONKEY[m], trials=lab, n=len(dd),
                                q25_ms=1000 * q[0], median_ms=1000 * q[1], q75_ms=1000 * q[2],
                                iqr_ms=1000 * (q[2] - q[0]), share_under_180ms_before_cut=under,
                                outside_150_500ms=float(((dd.reaction_time < 0.15) | (dd.reaction_time > 0.5)).mean())))
        ax = axes[i, j]
        bins = np.arange(0.1, 0.61, 0.01)
        ax.hist(dk[dk["correct_code_4.8deg"]].reaction_time, bins, alpha=0.6, label="correct")
        ax.hist(dk[~dk["correct_code_4.8deg"]].reaction_time, bins, alpha=0.6, label="incorrect")
        ax.set_xlim(0.15, 0.5); ax.set_title(f"reading {reading}, Monkey {MONKEY[m]}", fontsize=9)
axes[0, 0].legend(); axes[1, 0].set_xlabel("reaction time (s)"); plt.tight_layout(); plt.show()
rt = pd.DataFrame(rt_rows)
print(rt.round(3).to_string(index=False))
print("(Histograms show 150 to 500 ms; the share of trials outside that range is in column outside_150_500ms.)")
a = rt[rt.trials == "all"]
print("\nIQR above 10 ms and median within 150-500 ms, per reading and monkey:")
print(a.assign(holds=(a.iqr_ms > 10) & a.median_ms.between(150, 500))[["reading", "monkey", "iqr_ms", "median_ms", "holds"]].round(1).to_string(index=False))


**Result: reproduced.** Under both readings each monkey's distribution is broad (interquartile range 50.6 to 73.1 ms) with its median inside 150 to 500 ms (227 to 273 ms). Before the 180 ms cut, 1.7% (P) and 0.4% (E) of reading (a) trials fall under 180 ms. The cut-offs are the claims file's reading of "variable".

### ext1-misbinding-errors

> A common error is a saccade to the location of a non-target stimulus object, reporting the cued identity at the wrong location.

Source: Discussion, Position-identity binding, paragraph 1; Extended Data Fig. 1. Paper: "A common error in our task occurs when the monkey reports the identity of an object at the wrong location ( Extended Data Fig. 1 ), consistent with misbinding in working memory"

Criterion (claims file): Completed Triangle trials on the triangle with 2 or 3 objects, per monkey. Each response is assigned to the triangle vertex within the correctness radius (T2), as in load_triangle_data.py (response_location), and counted by target location and response location as in Extended Data Fig. 1. Among incorrect trials, the share whose response lies at a non-target object's vertex. The paper gives no number for "common"; two readings are run, (a) these responses are the largest class of errors (against responses at an empty vertex or at no vertex), and (b) they are more than half of the errors. Reproduced when both readings hold for both monkeys and both set sizes, not reproduced when neither does; when the readings give different outcomes the claim is methods_underspecified, with the shares reported.

In [ ]:
# ext1-misbinding-errors. Completed Triangle trials on the triangle with 2 or 3 objects; each
# response assigned to the vertex within the radius (load_triangle_data.py, response_location).
d = sel_a[sel_a.n_obj >= 2].copy()
ext1 = []
for rad, radius in wm.RADII.items():
    d["resp_loc"] = wm.response_location(d, radius)
    locs = d[["loc0", "loc1", "loc2"]].values
    tloc = locs[np.arange(len(d)), d.target.values]
    occupied = (locs == d.resp_loc.values[:, None]).any(1) & (d.resp_loc.values >= 0)
    cls = np.where(d[f"correct_{rad}"], "correct",
          np.where(d.resp_loc.values < 0, "no vertex",
          np.where(d.resp_loc.values == tloc, "target vertex, outside radius",
          np.where(occupied, "non-target object", "empty vertex"))))
    d["cls"] = cls
    for (m, n), g in d.groupby(["subject", "n_obj"]):
        e = g[g.cls != "correct"].cls.value_counts()
        share = e.get("non-target object", 0) / e.sum()
        largest = e.idxmax() == "non-target object"
        ext1.append(dict(radius=rad, monkey=MONKEY[m], n_obj=n, n_trials=len(g), n_errors=int(e.sum()),
                         non_target=int(e.get("non-target object", 0)), empty_vertex=int(e.get("empty vertex", 0)),
                         no_vertex=int(e.get("no vertex", 0)), target_outside=int(e.get("target vertex, outside radius", 0)),
                         share_non_target=share, a_largest=bool(largest), b_over_half=bool(share > 0.5)))
ext1 = pd.DataFrame(ext1)
print(ext1.round(3).to_string(index=False))
for rad in wm.RADII:
    e = ext1[ext1.radius == rad]
    print(f"{rad}: reading (a) holds in all four: {e.a_largest.all()}; reading (b): {e.b_over_half.all()}")
# Counts by target and response location, as Extended Data Fig. 1 plots them (code radius)
d["resp_loc"] = wm.response_location(d, wm.RADII["code_4.8deg"])
d["target_loc"] = d[["loc0", "loc1", "loc2"]].values[np.arange(len(d)), d.target.values]
print("\nTrials by target location (rows) and response location (columns, -1 = none), code radius:")
for (m, n), g in d.groupby(["subject", "n_obj"]):
    print(f"Monkey {MONKEY[m]}, {n} objects"); print(pd.crosstab(g.target_loc, g.resp_loc).to_string())


**Result: reproduced, resting on an assumption.** With the code's radius (0.2 arena widths), responses at a non-target object's vertex are the largest class of errors and more than half of them in both monkeys and set sizes (84.5% to 99.4%). With the Methods' radius (2.4 degrees) both readings fail for Monkey P with 2 objects (207 such errors against 249 at no vertex, 44.9%) and hold in the other three. The two radii give different outcomes; the code's radius is adopted because it is the one the task rewarded (data check: every completed trial under 0.2 rewarded, none above) and the one the paper's code uses to assign responses to vertices for Extended Data Fig. 1.

### fig5ab-ring-behavior

> Both monkeys perform well on the Ring dataset, with reaction-time variability like that on the Triangle dataset.

Source: Results, Replication of the Results with the Ring Dataset, paragraph 1; Figure 5a and 5b; Extended Data Fig. 9a and 9b (the Results cite "Extended Data Fig. 8a-b", but the per-monkey Ring figure is the ninth Extended Data legend). Paper: "Both monkeys performed well on the Ring dataset and exhibited reaction-time variability similar to the Triangle dataset"

Criterion (claims file): Completed Ring trials (T1), per monkey and pooled, fraction correct (T2) for 1 and 2 objects and the reaction-time distribution (T3), as figures/figure_5/performance_reaction_time/. The paper gives no number. Reproduced when the 2-object fraction correct is above chance (1/2) for both monkeys and each monkey's Ring reaction-time interquartile range is within a factor of 1.5 of its Triangle one; the factor is the agent's reading of "similar" and the ranges are reported. For reference, the paper's notebook printed P 0.958 and 0.903, E 0.945 and 0.756 for 1 and 2 objects with the code's radius.

In [ ]:
# fig5ab-ring-behavior. Completed Ring trials (T1), all conditions (Methods); the code's
# exclusion of the left/right trials shown as a variant.
R5 = Tp[Tp.task == "ring"].copy()
lr = (R5.n_obj == 2) & (np.isclose(R5.theta0, 0, atol=1e-3) | np.isclose(R5.theta0, 2*np.pi, atol=1e-3)) & np.isclose(R5.theta1, np.pi, atol=1e-3)
lr |= (R5.n_obj == 2) & np.isclose(R5.theta0, np.pi, atol=1e-3) & (np.isclose(R5.theta1, 0, atol=1e-3) | np.isclose(R5.theta1, 2*np.pi, atol=1e-3))
R5["left_right"] = lr
print("Left/right 2-object trials among completed Ring trials:", int(lr.sum()), "of", len(R5), f"({lr.mean():.3f})")
rows = []
for variant, d in [("all conditions", R5), ("without left/right", R5[~R5.left_right])]:
    for rad in wm.RADII:
        for m, dm in list(d.groupby("subject")) + [("pooled", d)]:
            for n in [1, 2]:
                g = dm[dm.n_obj == n]
                rows.append(dict(variant=variant, radius=rad, monkey=MONKEY.get(m, m), n_obj=n,
                                 n_trials=len(g), frac_correct=g[f"correct_{rad}"].mean()))
p5 = pd.DataFrame(rows)
print(p5.round(3).to_string(index=False))
rt5 = []
for m in ["Perle", "Elgar"]:
    q_r = np.percentile(R5[(R5.subject == m) & (R5.reaction_time >= 0.18)].reaction_time, [25, 50, 75])
    q_t = np.percentile(sel_a[(sel_a.subject == m) & (sel_a.reaction_time >= 0.18)].reaction_time, [25, 50, 75])
    rt5.append(dict(monkey=MONKEY[m], ring_median_ms=1000*q_r[1], ring_iqr_ms=1000*(q_r[2]-q_r[0]),
                    triangle_median_ms=1000*q_t[1], triangle_iqr_ms=1000*(q_t[2]-q_t[0]),
                    ratio=(q_r[2]-q_r[0])/(q_t[2]-q_t[0])))
rt5 = pd.DataFrame(rt5)
print("\nReaction times (trials >= 180 ms), Ring against Triangle (on-triangle trials):")
print(rt5.round(3).to_string(index=False))
for rad in wm.RADII:
    q = p5[(p5.variant == "all conditions") & (p5.radius == rad) & (p5.n_obj == 2) & p5.monkey.isin(["P", "E"])]
    print(f"{rad}: 2-object above 1/2 for both monkeys: {bool((q.frac_correct > 0.5).all())}")
print("IQR ratio within a factor of 1.5 for both monkeys:", bool(rt5.ratio.between(1/1.5, 1.5).all()))
fig, ax = plt.subplots(figsize=(4, 3))
for m in ["Perle", "Elgar"]:
    ax.hist(R5[(R5.subject == m) & (R5.reaction_time >= 0.18)].reaction_time, np.arange(0.1, 0.61, 0.01), alpha=0.5, label=MONKEY[m])
ax.set_xlim(0.15, 0.5); ax.set_xlabel("Ring reaction time (s)"); ax.legend(); plt.show()


**Result: reproduced.** The 2-object fraction correct on Ring trials is above 1/2 for both monkeys under both radii (code radius P 0.903, E 0.756; Methods radius P 0.786, E 0.721). The Ring interquartile range of reaction time is 0.935 (P) and 1.098 (E) times the Triangle one, inside the factor of 1.5. With the code's radius the 1-object values are P 0.960 and E 0.945 against the paper's notebook's 0.958 and 0.945, and the 2-object values match its 0.903 and 0.756.

## Units

### triangle-unit-counts

> The Triangle dataset holds 4,688 neurons, 2,937 in DMFC and 1,751 in FEF, from 24 sessions, 10 from Monkey P and 14 from Monkey E.

Source: Results, The Triangle Dataset, paragraph 1; Methods, Neurophysiology, last paragraph. Paper: "We recorded 4,688 well-isolated neurons (2,937 from DMFC and 1,751 from FEF) in the Triangle dataset across 24 sessions"

Criterion (claims file): Sessions are the counts stated in Methods, Neurophysiology, so no tolerance. Unit counts take the default 10%. Which units are counted is open: (a) good and mua units with delay-phase rate above 1 Hz on Triangle trials, which is how figures/figure_2/selectivity/selectivity.ipynb printed 4688, 2937 and 1751; (b) as (a) and also stable for at least 500 trials (T5, Methods); (c) good units only, reading "well-isolated" literally. Reproduced when all five values match under one reading. DMFC is probe s0, FEF the V-probes.

### ring-unit-counts

> The Ring dataset holds 3,051 neurons, 1,712 in DMFC and 1,339 in FEF, from 22 sessions, 10 from Monkey P and 12 from Monkey E.

Source: Results, Replication of the Results with the Ring Dataset, paragraph 1; Methods, Neurophysiology, last paragraph. Paper: "We recorded 3,051 well-isolated neurons (1,712 from DMFC and 1,339 from FEF) across 22 neurophysiology sessions"

Criterion (claims file): As triangle-unit-counts, with the same three readings, on Ring trials. The paper's Ring notebooks list 20 sessions (10 per monkey); the session dates found are reported against both counts.

In [ ]:
# triangle-unit-counts and ring-unit-counts. A unit's delay-phase mean rate is the mean over its
# completed stable trials of the session (T4, T5; run_cache_selectivity.py mean_firing_rate).
U = []
for x in S:
    if x["units"] is None:
        continue
    u = x["units"].copy()
    u["subject"], u["session"], u["task"] = x["subject"], x["session"], x["task"]
    u["delay_rate"] = np.nanmean(x["rates"]["delay"], 1)
    u["stim_rate"] = np.nanmean(x["rates"]["stimulus"], 1)
    u["n_completed_stable"] = np.isfinite(x["rates"]["delay"]).sum(1)
    u["dataset"] = "triangle" if x in tri else ("ring" if x in ring else "other")
    U.append(u)
U = pd.concat(U, ignore_index=True)
readings = {
    "a: good+mua, rate > 1 Hz": U.delay_rate > wm.MIN_RATE_HZ,
    "b: (a) and stable >= 500 trials": (U.delay_rate > wm.MIN_RATE_HZ) & (U.n_obs_trials >= wm.MIN_STABLE_TRIALS),
    "c: good only, rate > 1 Hz": (U.delay_rate > wm.MIN_RATE_HZ) & (U.quality == "good"),
}
cnt = []
for ds in ["triangle", "ring", "other"]:
    for name, m in readings.items():
        d = U[m & (U.dataset == ds)]
        cnt.append(dict(dataset=ds, reading=name, total=len(d), DMFC=(d.area == "DMFC").sum(), FEF=(d.area == "FEF").sum(),
                        P=(d.subject == "Perle").sum(), E=(d.subject == "Elgar").sum(),
                        sessions_P=d[d.subject == "Perle"].session.nunique(), sessions_E=d[d.subject == "Elgar"].session.nunique()))
    d = U[U.dataset == ds]
    cnt.append(dict(dataset=ds, reading="all sorted units", total=len(d), DMFC=(d.area == "DMFC").sum(), FEF=(d.area == "FEF").sum(),
                    P=(d.subject == "Perle").sum(), E=(d.subject == "Elgar").sum(),
                    sessions_P=d[d.subject == "Perle"].session.nunique(), sessions_E=d[d.subject == "Elgar"].session.nunique()))
cnt = pd.DataFrame(cnt)
print(cnt.to_string(index=False))
exp = {"triangle": (4688, 2937, 1751, 10, 14), "ring": (3051, 1712, 1339, 10, 12)}
for ds, (tot, dm, fe, sp, se) in exp.items():
    for _, r in cnt[(cnt.dataset == ds) & cnt.reading.str.match("[abc]:")].iterrows():
        ok = (abs(r.total - tot) <= round(0.1 * tot) and abs(r.DMFC - dm) <= round(0.1 * dm)
              and abs(r.FEF - fe) <= round(0.1 * fe) and r.sessions_P == sp and r.sessions_E == se)
        print(f"{ds}, {r.reading}: all five values match: {ok}")
print("Ring session dates:", {m: sorted(U[(U.dataset == 'ring') & (U.subject == m)].session.unique()) for m in MONKEY})
d6 = U[(U.dataset == "other") & (U.delay_rate > 1)]
print("Ring with Perle 2022-06-06 added, reading a:", int(cnt.query("dataset=='ring'").iloc[0].total) + len(d6), "units, P sessions 11")
code_ring = ["2022-06-07","2022-06-09","2022-06-10","2022-06-11","2022-06-12","2022-06-13","2022-06-14","2022-06-15","2022-06-16","2022-06-19",
             "2022-09-06","2022-09-07","2022-09-09","2022-09-10","2022-09-11","2022-09-12","2022-09-13","2022-09-14","2022-09-15","2022-09-20"]
d20 = U[(U.dataset == "ring") & U.session.isin(code_ring) & (U.delay_rate > 1)]
print("Ring, the 20 sessions of the code's model comparison, reading a:", len(d20), "units; DMFC", (d20.area == "DMFC").sum(), "FEF", (d20.area == "FEF").sum())


**Result: reproduced.** Reading (a), good and multi-units above 1 Hz in the delay phase, gives 4,688 units, 2,937 in DMFC and 1,751 in FEF, from 10 Perle and 14 Elgar sessions, the paper's numbers exactly. Reading (b), adding the Methods' 500-trial stability rule, gives 4,473, 2,857 and 1,616, also within tolerance. Reading (c), good units only, gives 1,563 and does not match: only 1,563 of the 4,688 "well-isolated neurons" are labelled single units.

**Result: reproduced.** Reading (a) gives 3,052 Ring units, 1,712 in DMFC and 1,340 in FEF, from 10 Perle and 12 Elgar sessions, against 3,051, 1,712 and 1,339; reading (b) gives 2,936, 1,655 and 1,281, also within tolerance; reading (c) gives 1,123 and does not match. The 22 sessions include Elgar 2022-09-18 and 2022-09-19, which the code's Ring model comparison leaves out (20 sessions, 2,862 units).

### ext2-stimulus-delay-activity

> Neurons are typically active in both the stimulus and the delay phase, and a unit's spatial preference is similar across the two phases.

Source: Results, The Triangle Dataset, paragraph 1; Extended Data Fig. 2a and 2b. Paper: "We found that neurons were typically active in both stimulus and delay phases ( Extended Data Fig. 2 )"

Criterion (claims file): Triangle units (T5), per monkey. Panel a, the stimulus-phase (0 to 1000 ms after stimulus onset) against the delay-phase mean rate (T4) per unit; panel b, per unit and triangle location, the 1-object mean rate at that location divided by the unit's mean over locations, stimulus phase against delay phase, for DMFC and FEF apart, as the legend defines it. The paper gives no test. Readings of "typically active in both": (a) the Pearson and (b) the Spearman correlation across units of panel a's rates is positive at p < 0.05, for both monkeys; (c) more than half of the units, all above 1 Hz in the delay phase by T5, are also above 1 Hz in the stimulus phase (the Methods' activity threshold applied to the other phase). Panel b is judged by readings (a) and (b) per monkey and area. Reproduced when all three readings hold for panel a in both monkeys and (a) and (b) hold for at least three of the four panel b plots; not reproduced when none hold; methods_underspecified when they differ. The second half of the statement is read from the title of the panel b legend, "Correlation in spatial selectivity across stimulus and delay phases"; the three-of-four rule is the agent's reading. The correlations and shares are reported.

In [ ]:
# ext2-stimulus-delay-activity. Triangle units above 1 Hz (T5). Panel a: stimulus against delay
# mean rate per unit; panel b: per unit and vertex, the 1-object rate at the vertex divided by
# the unit's mean over the three vertices, stimulus against delay (Extended Data Fig. 2 legend).
UT = U[(U.dataset == "triangle") & (U.delay_rate > wm.MIN_RATE_HZ)]
ra = []
for m, g in UT.groupby("subject"):
    pr, sp = st.pearsonr(g.stim_rate, g.delay_rate), st.spearmanr(g.stim_rate, g.delay_rate)
    ra.append(dict(monkey=MONKEY[m], n_units=len(g), pearson_r=pr[0], pearson_p=pr[1], spearman_rho=sp[0], spearman_p=sp[1],
                   share_stim_above_1Hz=float((g.stim_rate > 1).mean())))
ra = pd.DataFrame(ra)
print("Panel a:"); print(ra.to_string(index=False))
a_ok = bool(((ra.pearson_r > 0) & (ra.pearson_p < 0.05)).all() and ((ra.spearman_rho > 0) & (ra.spearman_p < 0.05)).all()
            and (ra.share_stim_above_1Hz > 0.5).all())
print("All three readings hold for panel a in both monkeys:", a_ok)
pb = []
for x in tri:
    tr = x["trials"]
    m = (tr.n_obj == 1).values & tr.on_triangle.values & tr.completed.values
    keep = np.nanmean(x["rates"]["delay"], 1) > wm.MIN_RATE_HZ
    for ph in ["stimulus", "delay"]:
        R = x["rates"][ph][keep][:, m]
        loc = tr.loc0.values[m]
        mu = np.stack([np.nanmean(R[:, loc == k], 1) for k in range(3)], 1)
        x[f"norm_{ph}"] = mu / mu.mean(1, keepdims=True)
    areas = x["units"].area.values[keep]
    for k in range(3):
        for i in range(keep.sum()):
            pb.append(dict(monkey=MONKEY[x["subject"]], area=areas[i], loc=k,
                           stim=x["norm_stimulus"][i, k], delay=x["norm_delay"][i, k]))
pb = pd.DataFrame(pb).replace([np.inf, -np.inf], np.nan).dropna()
rb = []
for (m, ar), g in pb.groupby(["monkey", "area"]):
    pr, sp = st.pearsonr(g.stim, g.delay), st.spearmanr(g.stim, g.delay)
    rb.append(dict(monkey=m, area=ar, n_points=len(g), pearson_r=pr[0], pearson_p=pr[1], spearman_rho=sp[0], spearman_p=sp[1]))
rb = pd.DataFrame(rb)
print("\nPanel b:"); print(rb.to_string(index=False))
nb_ok = int((((rb.pearson_r > 0) & (rb.pearson_p < 0.05)) & ((rb.spearman_rho > 0) & (rb.spearman_p < 0.05))).sum())
print(f"Panel b plots where readings (a) and (b) both hold: {nb_ok} of 4")
print("Points dropped from panel b (a unit with zero mean over vertices in a phase):",
      int(sum(3 * (np.nanmean(x['rates']['delay'], 1) > 1).sum() for x in tri) - len(pb)))
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, m in zip(axes, ["Perle", "Elgar"]):
    g = UT[UT.subject == m]
    for ar, col in [("DMFC", "tab:orange"), ("FEF", "tab:blue")]:
        h = g[g.area == ar]; ax.scatter(h.stim_rate, h.delay_rate, s=3, c=col, label=ar)
    ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlabel("stimulus rate (Hz)"); ax.set_ylabel("delay rate (Hz)")
    ax.set_title(f"Monkey {MONKEY[m]}"); ax.legend()
plt.show()
print("Ranges: stimulus", UT.stim_rate.min().round(3), "to", UT.stim_rate.max().round(1), "Hz; delay",
      UT.delay_rate.min().round(3), "to", UT.delay_rate.max().round(1), "Hz; units with a zero stimulus rate (not on log axes):", int((UT.stim_rate <= 0).sum()))


**Result: reproduced.** Stimulus- and delay-phase rates correlate across units (Pearson r 0.898 for P and 0.971 for E, Spearman 0.894 and 0.963, p below 1e-300), and 97.5% (P) and 99.1% (E) of units above 1 Hz in the delay are also above 1 Hz in the stimulus phase. The normalised location preferences of the two phases correlate positively in all four monkey and area plots (Pearson 0.398 to 0.600, Spearman 0.371 to 0.545, all p below 1e-89).

## Selectivity

AUCs of every unit above 1 Hz, used by fig2b-selectivity-range and fig5cd-ring-tuning, and the count of units entering the models (T7) used to estimate the model fits.

In [ ]:
# AUC (T6) of every unit above 1 Hz (T5) on 1-object trials: Triangle trials on the triangle
# labelled by vertex, Ring trials by nearest vertex (three 120-degree sectors); identity by
# fruit. Both p-value methods of T6. Own generator.
rng_auc = np.random.default_rng(620)
ID_NUM = {"a": 0, "b": 1, "c": 2}
auc_rows = []
for x in tri + ring:
    tr = x["trials"]; R = x["rates"]["delay"]
    m = (tr.n_obj == 1).values & tr.completed.values
    if x["task"] == "triangle":
        m &= tr.on_triangle.values
        pos = tr.loc0.values
    else:
        pos = wm.ring_sector(tr.theta0.values)
    idn = tr.id0.map(ID_NUM).fillna(-1).astype(int).values
    rate = np.nanmean(R, 1)
    for k in np.where(rate > wm.MIN_RATE_HZ)[0]:
        r = R[k]; ok = m & np.isfinite(r)
        row = dict(subject=x["subject"], session=x["session"], dataset=x["task"], unit_index=k,
                   area=x["units"].area.iloc[k], quality=x["units"].quality.iloc[k],
                   n_obs_trials=x["units"].n_obs_trials.iloc[k], delay_rate=rate[k], n_1obj=int(ok.sum()))
        for meth in ["methods", "code"]:
            row[f"pos_auc_{meth}"], row[f"pos_p_{meth}"] = wm.unit_auc(r[ok], pos[ok], rng_auc, meth)
            row[f"id_auc_{meth}"], row[f"id_p_{meth}"] = wm.unit_auc(r[ok], idn[ok], rng_auc, meth)
        auc_rows.append(row)
AUC = pd.DataFrame(auc_rows)
for meth in ["methods", "code"]:
    AUC[f"modeled_{meth}"] = (((AUC[f"pos_auc_{meth}"] > 0.65) & (AUC[f"pos_p_{meth}"] < 0.05))
                              | ((AUC[f"id_auc_{meth}"] > 0.65) & (AUC[f"id_p_{meth}"] < 0.05)))
print("Units with AUCs:", AUC.groupby(["dataset", "subject"]).size().to_dict())
print("Fewest 1-object trials of a unit:", AUC.n_1obj.min())
nm = AUC.groupby(["dataset", "subject", "session"])[["modeled_methods", "modeled_code"]].sum().reset_index()
print("\nUnits entering the models per session (T7: AUC > 0.65 with p < 0.05, rate > 1 Hz):")
print(nm.to_string(index=False))


### fig2b-selectivity-range

> Delay-phase position and identity AUCs on 1-object trials span a wide range across units.

Source: Results, The Triangle Dataset, paragraph 1; Figure 2b. Paper: "A Receiver Operating Characteristic analysis of firing rates in the delay phase revealed a wide range of selectivities for position and identity across neurons ( Fig. 2b )"

Criterion (claims file): Position and identity AUC (T6) of every Triangle unit (T5) per monkey, with significance at p < 0.05 as the legend shades it. The paper gives no number for "wide". Reproduced when, for both monkeys and both features, the AUCs run from near 0.5 (lowest decile below 0.6) to high values (highest decile above 0.7), with both significant and non-significant units present; the deciles are reported under both p-value methods of T6. These cut-offs are the agent's reading of "wide range" and are recorded as such.

In [ ]:
# fig2b-selectivity-range: deciles of position and identity AUC per monkey, Triangle units.
def auc_range(A, label):
    rows = []
    for meth in ["methods", "code"]:
        for m, g in list(A.groupby("subject")) + ([("pooled", A)] if label == "ring" else []):
            sig = (g[f"pos_p_{meth}"] < 0.05) | (g[f"id_p_{meth}"] < 0.05)
            for feat in ["pos", "id"]:
                v = g[f"{feat}_auc_{meth}"].dropna()
                rows.append(dict(p_method=meth, monkey=MONKEY.get(m, m), feature=feat, n_units=len(v),
                                 min=v.min(), p10=np.percentile(v, 10), median=np.median(v), p90=np.percentile(v, 90), max=v.max(),
                                 n_significant=int((g[f"{feat}_p_{meth}"] < 0.05).sum()),
                                 n_shaded_significant=int(sig.sum()), n_shaded_not=int((~sig).sum())))
    r = pd.DataFrame(rows)
    r["holds"] = (r.p10 < 0.6) & (r.p90 > 0.7) & (r.n_shaded_significant > 0) & (r.n_shaded_not > 0)
    return r
for variant, A in [("units above 1 Hz", AUC[AUC.dataset == "triangle"]),
                   ("also stable >= 500 trials", AUC[(AUC.dataset == "triangle") & (AUC.n_obs_trials >= 500)])]:
    r = auc_range(A, "triangle")
    print(f"\nTriangle, {variant}:"); print(r.round(3).to_string(index=False))
    print("Holds for both monkeys and both features:", r.groupby("p_method").holds.all().to_dict())
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, m in zip(axes, ["Perle", "Elgar"]):
    g = AUC[(AUC.dataset == "triangle") & (AUC.subject == m)]
    sig = (g.pos_p_methods < 0.05) | (g.id_p_methods < 0.05)
    ax.scatter(g.pos_auc_methods[~sig], g.id_auc_methods[~sig], s=4, c="0.8")
    ax.scatter(g.pos_auc_methods[sig], g.id_auc_methods[sig], s=4, c="k")
    ax.set_xlim(0.49, 1.01); ax.set_ylim(0.49, 1.01); ax.set_title(f"Monkey {MONKEY[m]}, N={len(g)}")
    ax.set_xlabel("position AUC"); ax.set_ylabel("identity AUC")
plt.show()
print("All AUCs lie in [0.5, 1]:", bool(AUC.filter(like="_auc_").stack().between(0.5, 1).all()))


**Result: partially reproduced.** Position AUCs span a wide range in both monkeys under both p-value methods (lowest decile 0.538 to 0.574, highest decile 0.706 to 0.897), with significant and non-significant units present. Identity AUCs do not reach the claims file's upper cut-off: their highest decile is 0.654 (E) and 0.700 (P) with the Methods p-value, and 0.681 (E) and 0.715 (P) with the code's, so the rule holds for identity only for Monkey P under the code's p-value. Adding the 500-trial rule changes no conclusion. Identity AUCs still run from 0.50 to 0.97 or more in both monkeys.

### fig5cd-ring-tuning

> Ring neurons in DMFC and FEF show a variety of spatial receptive fields and diverse delay-phase position and identity selectivities, as in the Triangle dataset.

Source: Results, Replication of the Results with the Ring Dataset, paragraph 1; Figure 5c and 5d; Extended Data Fig. 9c. Paper: "These neurons showed a variety of spatial receptive fields in both DMFC and FEF ( Fig. 5c ; Extended Data Fig. 9c ) and diverse selectivities to position and identity in delay-phase firing rates ( Fig. 5d ), similar to the Triangle dataset"

Criterion (claims file): Position and identity AUC (T6) of Ring units, judged as fig2b-selectivity-range. Tuning curves as Methods, Ring Dataset Tuning Curves (256 points on the ring, 100 bootstraps of 100 1-object trials, Gaussian kernel of 2.4 degrees); "variety" is judged by the preferred angles of units with significant position AUC, reproduced when in each area they fall in all three 120-degree sectors.

In [ ]:
# fig5cd-ring-tuning. Ring AUC judged as fig2b, per monkey and pooled; tuning curves as Methods,
# Ring Dataset Tuning Curves (256 points on the ring, 100 bootstraps of 100 1-object trials,
# Gaussian kernel sigma 0.1 arena widths = 2.4 deg). Preferred angle = peak of the mean curve.
r5 = auc_range(AUC[AUC.dataset == "ring"], "ring")
print("Ring AUC deciles:"); print(r5.round(3).to_string(index=False))
print("Holds for both monkeys and both features:", r5[r5.monkey != "pooled"].groupby("p_method").holds.all().to_dict())
rng_tc = np.random.default_rng(5)
ang = np.linspace(0, 2 * np.pi, 256, endpoint=False)
P = wm.RADIUS * np.stack([np.cos(ang), np.sin(ang)], 1)
pref = []
for x in ring:
    tr = x["trials"]
    m = (tr.n_obj == 1).values & tr.completed.values
    keep = np.where(np.nanmean(x["rates"]["delay"], 1) > wm.MIN_RATE_HZ)[0]
    xy = np.stack([tr.x0.values, tr.y0.values], 1)
    for k in keep:
        r = x["rates"]["delay"][k]; ok = m & np.isfinite(r)
        rr, pp = r[ok], xy[ok]
        K = np.exp(-((P[:, None, :] - pp[None]) ** 2).sum(-1) / (2 * 0.1 ** 2))   # 256 x n
        idx = rng_tc.integers(0, len(rr), size=(100, 100))
        curves = np.stack([(K[:, i] @ rr[i]) / K[:, i].sum(1) for i in idx])
        mc = curves.mean(0)
        pref.append(dict(subject=x["subject"], session=x["session"], unit_index=k, area=x["units"].area.iloc[k],
                         pref_angle=ang[np.argmax(mc)], depth_of_tuning=(mc.max() - mc.min()) / mc.mean()))
pref = pd.DataFrame(pref).merge(AUC[AUC.dataset == "ring"][["session", "unit_index", "pos_p_methods", "pos_p_code"]], on=["session", "unit_index"])
pref["sector"] = wm.ring_sector(pref.pref_angle.values)
for meth in ["methods", "code"]:
    sg = pref[pref[f"pos_p_{meth}"] < 0.05]
    tab = pd.crosstab([sg.subject, sg.area], sg.sector)
    print(f"\nPreferred-angle sector (0 right, 1 top-left, 2 bottom-left) of units with significant position AUC, {meth} p:")
    print(tab.to_string())
    tab2 = pd.crosstab(sg.area, sg.sector)
    print("Every area has units in all three sectors (pooled over monkeys):", bool((tab2 > 0).all().all() and tab2.shape[1] == 3),
          "; in each monkey and area:", bool((tab > 0).all().all() and tab.shape[1] == 3))
fig, axes = plt.subplots(1, 2, figsize=(7, 3), subplot_kw=dict(projection="polar"))
for ax, ar in zip(axes, ["DMFC", "FEF"]):
    sg = pref[(pref.pos_p_methods < 0.05) & (pref.area == ar)]
    ax.hist(sg.pref_angle, bins=24); ax.set_title(ar)
plt.show()


**Result: partially reproduced.** Units with significant position AUC have preferred angles in all three sectors in each area and each monkey under both p-value methods (fewest per cell 48), so the variety of receptive fields holds. The AUC part judged as fig2b fails: identity AUCs have a highest decile of 0.611 to 0.653, and Elgar's position AUCs 0.658 (Methods p) and 0.674 (code p), below 0.7; Perle's position AUCs (0.796, 0.805) and the pooled ones (0.740, 0.751) pass.

## Noise correlations

### fig2d-signal-noise-triangle

> In the Triangle dataset, pairs of neurons with similar 1-object tuning have positive noise correlations on 2-object trials, and dissimilarly tuned pairs negative ones.

Source: Results, The Triangle Dataset, paragraph 2; Figure 2d. Paper: "Pairs of neurons with similar selectivities on 1-object trials tended to have positive noise correlations on 2-object trials ... Conversely, dissimilarly tuned pairs of neurons tended to have negative noise correlations."

Criterion (claims file): Pairs of simultaneously recorded Triangle units, per monkey, delay-phase rates (T4). Main method, Methods, Noise Correlations: signal correlation is the correlation across the 9 1-object conditions of the two units' mean rates, pairs dropped unless each condition has at least 3 trials; noise correlation is the mean over the 18 2-object conditions (identity included) with at least 3 shared trials of the across-trial correlation. Variant, the paper's figures/figure_2/noise_correlations/run_compute_triangle.py: per pair and per triple of two 1-object conditions and their 2-object composite, the signal correlation from the two 1-object conditions with trials shuffled between units, the noise correlation on the composite, units above 1 Hz, rows kept with more than 5 trials in each condition. As in plot_noise_vs_signal_correlation.ipynb, signal correlation is cut into 6 bins with edges at linspace(-0.8, 0.8, 7), ends open, and the mean noise correlation per bin is shown with its 95% confidence interval. The paper gives no test. Reproduced when, for both monkeys, the bin of highest signal correlation has a mean noise correlation above zero and the lowest bin one below zero, each with a 95% interval excluding zero; partially reproduced when only the rise from the lowest to the highest bin holds (non-overlapping intervals). Both methods are run; when they differ the claim is methods_underspecified.

In [ ]:
# fig2d-signal-noise-triangle. Main method (Methods, Noise Correlations) and the code's variant;
# 6 bins with edges at linspace(-0.8, 0.8, 7), ends open; mean noise correlation per bin with
# a 95% interval, per monkey.
rng_nc = np.random.default_rng(2)
edges6 = np.linspace(-0.8, 0.8, 7)
NCm, NCc = [], []
for x in tri:
    d, ncond = wm.triangle_noise_methods(x)
    d["subject"], d["session"] = x["subject"], x["session"]; NCm.append(d)
    d = wm.triangle_noise_code(x, rng_nc)
    d["subject"], d["session"] = x["subject"], x["session"]; NCc.append(d)
NCm, NCc = pd.concat(NCm, ignore_index=True), pd.concat(NCc, ignore_index=True)
print("Pairs (Methods):", NCm.groupby("subject").size().to_dict(), "; rows (code):", NCc.groupby("subject").size().to_dict())
print("Pairs with a finite noise correlation (Methods):", int(np.isfinite(NCm.noise).sum()), "of", len(NCm))
def judge_bins(b):
    hi, lo = b.iloc[-1], b.iloc[0]
    full = hi.mean_noise > 0 and hi.ci_lo > 0 and lo.mean_noise < 0 and lo.ci_hi < 0
    rise = hi.ci_lo > lo.ci_hi
    return full, rise
res2d = {}
fig, axes = plt.subplots(1, 2, figsize=(8, 3), sharey=True)
for name, D, col in [("Methods", NCm, "noise"), ("code", NCc, "noise")]:
    for ax, m in zip(axes, ["Perle", "Elgar"]):
        b = wm.bin_means(D[D.subject == m].signal, D[D.subject == m][col], edges6)
        res2d[(name, MONKEY[m])] = judge_bins(b)
        print(f"\n{name}, Monkey {MONKEY[m]}:"); print(b.round(4).to_string(index=False))
        ax.errorbar(np.arange(6), b.mean_noise, yerr=[b.mean_noise - b.ci_lo, b.ci_hi - b.mean_noise], marker="o", label=name)
        ax.axhline(0, c="k", lw=0.5); ax.set_title(f"Monkey {MONKEY[m]}"); ax.set_xlabel("signal-correlation bin")
axes[0].set_ylabel("mean noise correlation"); axes[0].legend(); plt.show()
print("\n(highest bin > 0 and lowest < 0 with intervals excluding 0, rise with non-overlapping intervals):")
for k, v in res2d.items():
    print(k, v)
# per session, Methods: lowest- and highest-bin means
ps = []
for (m, s), g in NCm.groupby(["subject", "session"]):
    b = wm.bin_means(g.signal, g.noise, edges6)
    ps.append(dict(monkey=MONKEY[m], session=s, pairs=len(g), lowest_bin=b.mean_noise.iloc[0], highest_bin=b.mean_noise.iloc[-1],
                   pearson_signal_noise=st.pearsonr(*g[["signal", "noise"]].dropna().values.T)[0]))
print(pd.DataFrame(ps).round(3).to_string(index=False))

# Code variant with intervals clustered by pair: a pair contributes up to 18 rows (one per
# triple of conditions), so rows are not independent; the cluster-robust interval counts each
# pair once (wm.bin_means_clustered).
NCc["pair"] = NCc.session + ":" + NCc.unit_0.astype(str) + "-" + NCc.unit_1.astype(str)
print("\nCode variant, rows per pair: median", int(NCc.groupby("pair").size().median()), ", max", int(NCc.groupby("pair").size().max()))
res2d_cl = {}
for m in ["Perle", "Elgar"]:
    g = NCc[NCc.subject == m]
    b = wm.bin_means_clustered(g.signal, g.noise, g.pair, edges6)
    res2d_cl[MONKEY[m]] = judge_bins(b)
    print(f"Code variant, Monkey {MONKEY[m]}, intervals clustered by pair:"); print(b.round(4).to_string(index=False))
print("(full, rise) with intervals clustered by pair, code variant:", res2d_cl)


**Result: blocked (methods_underspecified).** The two methods give different outcomes, which the criterion settles as underspecified. With the Methods' signal correlation (across the 9 1-object condition means), both monkeys show negative noise correlation in the lowest bin (P -0.0041, interval -0.0071 to -0.0011; E -0.0230, -0.0244 to -0.0216) and positive in the highest (P 0.0574; E 0.0490), so that reading reproduces the claim. With the code's signal correlation (two 1-object conditions, trials shuffled), Monkey P does too (-0.0189 and 0.0769), but Monkey E's lowest bin, 124 rows, is 0.0245 (interval -0.0296 to 0.0786); only the rise from lowest to highest bin holds (0.1412, interval 0.0975 to 0.1849). Signal and noise correlation correlate positively in every session (Pearson 0.100 to 0.442). Counting each pair once, with intervals clustered by pair (a pair has up to 18 rows in the code variant), changes nothing: Monkey E's lowest bin holds 124 rows from 98 pairs, 0.0245 (interval -0.0327 to 0.0818), its highest 0.1412 (0.0948 to 0.1877), and Monkey P's lowest bin stays below zero (-0.0309 to -0.0068).

### fig5e-signal-noise-ring

> In the Ring dataset, noise correlation rises with signal correlation, as in the Triangle dataset.

Source: Results, Replication of the Results with the Ring Dataset, paragraph 1; Figure 5e; Extended Data Fig. 9d. Paper: "We also found a positive relationship between signal correlation and noise correlation ( Fig. 5e ; Extended Data Fig. 9d ), corroborating our findings in the Triangle dataset"

Criterion (claims file): As Methods, Ring Dataset Signal and Noise Correlations (the Methods call it Fig. 5c), on pairs of Ring units: trials embedded in 6 dimensions (x and y of each fruit, 0 when absent); signal correlation on 1-object trials between one unit's rate and the other's on the nearest-neighbour trial. The noise correlation is open between (a) the Methods' wording, per 2-object trial the correlation of the two units over its 10 nearest-neighbour trials, averaged over trials, and (b) the paper's figures/figure_5/noise_correlations/run_compute_ring.py, each trial's rate minus the mean of its 10 neighbours, one Pearson correlation of these residuals across trials. Pairs kept with at least 50 1-object and 200 2-object trials (noise_correlations.ipynb); 8 bins of signal correlation with edges at linspace(-s, s, 9), s 0.8 for P and pooled, 0.3 for E. The paper gives no test. Reproduced when, per monkey, the mean noise correlation of the highest bin exceeds that of the lowest with non-overlapping 95% intervals; when the readings differ the claim is methods_underspecified.

In [ ]:
# fig5e-signal-noise-ring. Ordered pairs of Ring units above 1 Hz (wm.ring_noise); 8 bins with
# edges at linspace(-s, s, 9), s = 0.8 for P and pooled, 0.3 for E (noise_correlations.ipynb).
NR = []
for x in ring:
    d = wm.ring_noise(x)
    d["subject"], d["session"] = x["subject"], x["session"]; NR.append(d)
NR = pd.concat(NR, ignore_index=True)
print("Ordered pairs kept:", NR.groupby("subject").size().to_dict(), "; sessions with pairs:", NR.session.nunique())
res5e = {}
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
for ax, (m, s) in zip(axes, [("Perle", 0.8), ("Elgar", 0.3), ("pooled", 0.8)]):
    g = NR if m == "pooled" else NR[NR.subject == m]
    edges8 = np.linspace(-s, s, 9)
    for col, lab in [("noise_a", "a: Methods"), ("noise_b", "b: code")]:
        b = wm.bin_means(g.signal, g[col], edges8)
        hi, lo = b.iloc[-1], b.iloc[0]
        res5e[(MONKEY.get(m, m), lab)] = bool(hi.mean_noise > lo.mean_noise and hi.ci_lo > lo.ci_hi)
        print(f"\n{lab}, {m}:"); print(b.round(4).to_string(index=False))
        ax.errorbar(np.arange(8), b.mean_noise, yerr=[b.mean_noise - b.ci_lo, b.ci_hi - b.mean_noise], marker="o", label=lab)
    ax.set_title(m); ax.axhline(0, c="k", lw=0.5); ax.set_xlabel("signal-correlation bin")
axes[0].legend(); plt.show()
print("\nHighest bin above lowest with non-overlapping intervals:", res5e)
print("Signal-noise Pearson correlation across pairs:", {m: round(st.pearsonr(*g[["signal", "noise_b"]].dropna().values.T)[0], 3) for m, g in NR.groupby("subject")})

# run_compute_ring.py loops over ordered pairs (unit_0, then every other unit_1), so each
# unordered pair appears twice; intervals clustered by unordered pair (wm.bin_means_clustered).
NR["pair"] = (NR.session + ":" + np.minimum(NR.unit_0, NR.unit_1).astype(str) + "-"
              + np.maximum(NR.unit_0, NR.unit_1).astype(str))
print("\nRows per unordered pair:", NR.groupby("pair").size().value_counts().to_dict())
res5e_cl = {}
for m, s in [("Perle", 0.8), ("Elgar", 0.3), ("pooled", 0.8)]:
    g = NR if m == "pooled" else NR[NR.subject == m]
    edges8 = np.linspace(-s, s, 9)
    for col, lab in [("noise_a", "a: Methods"), ("noise_b", "b: code")]:
        b = wm.bin_means_clustered(g.signal, g[col], g.pair, edges8)
        hi, lo = b.iloc[-1], b.iloc[0]
        res5e_cl[(MONKEY.get(m, m), lab)] = bool(hi.mean_noise > lo.mean_noise and hi.ci_lo > lo.ci_hi)
        print(f"{lab}, {m}, lowest and highest bins, intervals clustered by pair:"); print(b.iloc[[0, -1]].round(4).to_string(index=False))
print("Highest bin above lowest with non-overlapping intervals clustered by pair:", res5e_cl)


**Result: partially reproduced.** Both noise-correlation readings agree. For Monkey P, and pooled, the highest signal-correlation bin has a higher mean noise correlation than the lowest with non-overlapping intervals (P, reading (a), 0.1728 against -0.1083; reading (b), 0.1620 against -0.0629). For Monkey E, on the claims file's narrower bins (s = 0.3), the highest bin (0.0238, interval 0.0151 to 0.0324, reading (a)) overlaps the lowest (0.0174, 0.0077 to 0.0271); its means rise over the middle bins (0.0145 to 0.0214), and the across-pair correlation of signal and noise is 0.017 for E and 0.061 for P. The code's pairs are ordered, so each unordered pair appears twice (224,583 pairs, two rows each); with intervals clustered by pair, E's highest bin (0.0238, 0.0131 to 0.0344) still overlaps its lowest (0.0174, 0.0059 to 0.0288), and P's bins stay apart (0.1728, 0.0873 to 0.2584, against -0.1083, -0.1504 to -0.0663, reading (a)).

## Model fitting: the cost of the paper's encoding models

The claims fig4a-gain-best-triangle, ext6-gain-best-areas, ext5-general-slot, fig4b-identifiability, nslots-two-best, ext11-kappa-tuning, ext7-gains-distributed, fig4c-gain-performance-triangle, fig4d-gain-rt-triangle, ext8-slot-switching-behavior, fig5f-gain-best-ring, fig5g-gain-performance-ring and fig5h-gain-rt-ring all rest on fitting the Slot, Switching and Gain models (T7, T8). The cell below times one gradient step at the paper's sizes and estimates the whole; the fits are not run.

In [ ]:
# Model-fitting claims (fig4a and everything resting on fitted models): time one gradient step
# of an MLP the size the paper uses, in numpy on this machine, and estimate the whole.
# Sizes from Methods, Modeling, and the paper's phys_modeling/configs/neural/triangle_gain.py
# (also ring_gain.py, ring_slot.py): hidden_sizes=(512, 1024, 512), batch_size=4096,
# training_steps=10000; the batch is 4,096 (trial, time-bin) samples, up to 3 objects per sample, each
# through an embedding MLP (input 18) and the baseline MLP (input 5); output one value per
# modeled unit. The step count is that of Methods, Tuning the Temporal Smoothness and Training
# Steps (kappa 0 to 200 ms by 20 ms = 11 values, two splits, 10,000 steps each) and Model-Fitting
# (10 splits, up to 10,000 steps). This is a lower bound: it leaves out the Slot model's
# loss over every slot and assignment, evaluation, and the data pipeline.
rng_mlp = np.random.default_rng(7)
def mlp_step(n_in, n_out, rows, sizes=(512, 1024, 512)):
    dims = (n_in,) + sizes + (n_out,)
    Ws = [rng_mlp.normal(0, 0.05, (a, b)).astype(np.float32) for a, b in zip(dims[:-1], dims[1:])]
    x = rng_mlp.normal(size=(rows, n_in)).astype(np.float32)
    acts = [x]
    for W in Ws[:-1]:
        acts.append(np.maximum(acts[-1] @ W, 0))
    out = acts[-1] @ Ws[-1]
    g = np.ones_like(out) / out.size
    for i in range(len(Ws) - 1, -1, -1):
        gW = acts[i].T @ g
        g = (g @ Ws[i].T) * (acts[i] > 0)
    return out
n_units = int(np.median(AUC[AUC.dataset == "triangle"].groupby("session").modeled_methods.sum()))
import time as _time
t0 = _time.perf_counter()
for _ in range(5):
    mlp_step(18, n_units, 4096 * 3); mlp_step(5, n_units, 4096 * 3)
step_s = (_time.perf_counter() - t0) / 5
steps_per_session_model = 11 * 2 * 10000 + 10 * 10000
est = {
    "fig4a, ext11 (24 Triangle sessions x 3 models)": 24 * 3 * steps_per_session_model,
    "ext5 (24 sessions, General Slot)": 24 * 1 * steps_per_session_model,
    "ext6 (8 sessions x 3 restrictions x 3 models)": 8 * 3 * 3 * steps_per_session_model,
    "fig4b (10 sessions x 3 generating x 3 fitted models, 11 kappas x 2 splits + 2 to 5 splits)": 10 * 9 * (11 * 2 * 10000 + 2 * 10000),
    "nslots (2 datasets x 2 monkeys x 4 sessions x 4 slot counts x 2 splits)": 2 * 2 * 4 * 4 * 2 * 10000,
    "fig5f (22 Ring sessions x 3 models)": 22 * 3 * steps_per_session_model,
}
print(f"Median modeled units per Triangle session: {n_units}; one gradient step (forward and backward of both MLPs): {step_s:.3f} s")
print(f"One fit of 10,000 steps: {10000 * step_s / 3600:.1f} h")
for k, v in est.items():
    print(f"{k}: {v:,} steps, {v * step_s / 3600 / 24:,.0f} days on this machine")

# Faster hardware: this machine has no GPU (printed below). As an allowance for one, the
# estimates are also given 100 times faster, to compare with the 60-minute notebook limit.
import shutil, os as _os
print("GPU on this machine (nvidia-smi found):", shutil.which("nvidia-smi") is not None, "; CPU cores:", _os.cpu_count())
for k, v in est.items():
    print(f"{k}, 100 times faster: {v * step_s / 100 / 3600:,.1f} h")


**Result: blocked (too_large) for all 13 model-fitting claims.** One gradient step of the paper's models at its sizes takes about 1.0 s here (0.983 s, printed above; the timing varies with the machine's load), so one fit of 10,000 steps takes about 2.7 h, and the kappa sweep and ten splits of fig4a-gain-best-triangle alone come to 23,040,000 steps, about 262 days on this machine. The Ring comparison (about 240 days), the area and single-unit controls (about 262 days), the General Slot model (about 87 days), the identifiability test (about 246 days) and the slot-count sweep (about 15 days) are each far over the notebook limit and the run's time. The behavioural regressions on the gains (fig4c, fig4d, fig5g, fig5h), the distribution of gains (ext7) and the Slot and Switching attention (ext8) need these fits and are blocked with them. This machine has no GPU; allowing for one by a speed-up of 100 times still leaves 3.5 h (slot-count sweep) to 62.9 h (fig4a), over the 60-minute notebook limit. None of the fits was started.

In [ ]:
print(f"Notebook ran in {(time.time() - t_start) / 60:.1f} min")


## Summary

| claim | outcome |
|---|---|
| fig1b-performance-load | reproduced |
| fig1c-reaction-times | reproduced |
| ext1-misbinding-errors | reproduced (assumed: correctness radius 0.2 arena widths) |
| triangle-unit-counts | reproduced |
| ring-unit-counts | reproduced |
| ext2-stimulus-delay-activity | reproduced |
| fig2b-selectivity-range | partially reproduced |
| fig5ab-ring-behavior | reproduced |
| fig5cd-ring-tuning | partially reproduced |
| fig2d-signal-noise-triangle | blocked, methods_underspecified |
| fig5e-signal-noise-ring | partially reproduced |
| ext3ab-decoding | partially reproduced (decoding notebook) |
| ext3c-decoder-orthogonality | partially reproduced (decoding notebook) |
| fig4a, ext6, ext5, fig4b, nslots, ext11, ext7, fig4c, fig4d, ext8, fig5f, fig5g, fig5h | blocked, too_large |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The encoding models were not fitted (one fit is 2.4 h here; the comparison is hundreds of days). AUCs use mid-ranks for ties, and both p-value methods (the Methods' 1,000 label shuffles and the code's 5,000 resampled labels) are shown; the code-p counts of units entering the models run above the counts the code's selectivity notebook printed for Elgar sessions (that notebook's output is not reproduced here), while the Methods p picks the same decoding sessions as the code. The code's Triangle noise-correlation variant shuffles trials for all units at once and takes each pair's shared trials after the shuffle. Ring noise correlations use each pair's shared stable trials, with neighbours taken from one sort per session; ties among neighbours are broken by trial order. Intervals of means are normal 95% intervals; for the code's Triangle variant and the Ring pairs, which give a pair several rows, intervals clustered by pair are also shown. Tuning-curve preferred angles are the peaks of the mean of the 100 bootstrapped curves.

What was found about the dataset. The trials and units tables hold everything the claims tested here use, and the unit counts of the paper are recovered exactly (Triangle) or to one unit (Ring). The task rewarded responses within 0.2 arena widths (4.8 degrees) of the target, not the 2.4 degrees of the Methods. The cue phase lasts 250 ms, not 180 ms, and `response_time` (described as the response time from eye tracking) is earlier than the response phase in 47.9% of completed trials and before cue onset in some, so it reads as saccade onset, which its description does not say. The electrodes tables give no brain area (all "unknown"). Two Perle sessions are not among the paper's (2022-06-06, 2-object trials only, with spike sorting; 2022-06-08, behaviour only) and are not described. Of the paper's "well-isolated neurons", two thirds are labelled multi-unit.
